# Chapter 12 — Build It Yourself: The KV-Cache

See why naive generation wastes work, **implement the cache** on a single attention layer, watch
a real GPT generate identically-but-faster with it, and price the cache's memory.

> 💡 **No GPU needed** — generation is just forward passes. Running a ✍️ cell before you fill it
> prints a friendly ⚠️, not an error.

In [ ]:
import torch
import torch.nn as nn
print('ready')

## Step 1 — The waste, in numbers  ▶️ Run this

To generate token *t*, the naive loop recomputes the keys/values of **all** *t* tokens so far —
then keeps only the last one's prediction. Over a whole run that's `1+2+...+T` recomputations. The
cache computes each token's k/v **once**. Run it and see the gap explode with length.

In [ ]:
for n in [10, 100, 1000]:
    naive = sum(range(1, n + 1))      # step t recomputes t key/value pairs
    print(f'{n:>4} tokens:  naive recomputes {naive:>7,} k/v pairs  |  cached computes {n:>5,}  ->  {naive//n}x more work')

## Step 2 — Implement the cache  ✍️ Your turn

The whole trick: instead of recomputing every past key/value, keep a running **cache** and append
only the new token's. We'll prove that attention computed *from the cache* equals attention
computed from scratch. Fill the two append lines (handle the first token, when the cache is empty).

<details><summary>Stuck? Show the answer</summary>

```python
cache_k = torch.cat([cache_k, kt]) if cache_k is not None else kt
cache_v = torch.cat([cache_v, vt]) if cache_v is not None else vt
```
</details>

In [ ]:
torch.manual_seed(0)
D = 16
Wq, Wk, Wv = (nn.Linear(D, D, bias=False) for _ in range(3))
seq = torch.randn(6, D)                      # 6 token 'embeddings'

# FULL attention for the last token — recompute k/v for ALL tokens (the naive answer):
K_all, V_all, q_last = Wk(seq), Wv(seq), Wq(seq[-1:])
full = torch.softmax(q_last @ K_all.T / D**0.5, -1) @ V_all

# CACHED — process one token at a time, never recomputing the past:
cache_k, cache_v = None, None
for t in range(6):
    kt, vt = Wk(seq[t:t+1]), Wv(seq[t:t+1])  # only THIS token's k, v
    # ✍️ append kt, vt to the cache
    cache_k = None      # replace
    cache_v = None      # replace

cached = None
if cache_k is not None:
    cached = torch.softmax(q_last @ cache_k.T / D**0.5, -1) @ cache_v
print('cache filled:', cache_k is not None)

In [ ]:
# ▶️ Check your work
try:
    if cache_k is None or cached is None:
        print('⚠️  Fill in the ✍️ cell above (append kt, vt to the cache), then re-run. (Expected until you do.)')
    elif torch.allclose(full, cached, atol=1e-6):
        print('✅ Cached attention == full attention (identical). The cache stores exactly what naive recomputes.')
    else:
        print('⚠️  The cached result differs from full attention — check the append order (past, then new).')
except Exception as e:
    print('❌', e)

## Step 3 — A real GPT: identical output, real speedup  ▶️ Run this

Now the payoff on a full (small) GPT with two generators — `generate_naive` (re-runs everything)
and `generate_cached` (prefill + decode with the cache). They produce the **same** tokens; the
cache is just faster, and the gap grows with length. (The next cell defines the model — **just
run it, no editing**; the same cache idea from Step 2 lives inside its attention.)

In [ ]:
import math, time
VOCAB, C, NH, NL, BLOCK = 65, 128, 4, 3, 512; HD = C // NH
class Attn(nn.Module):
    def __init__(s): super().__init__(); s.q=nn.Linear(C,C); s.k=nn.Linear(C,C); s.v=nn.Linear(C,C); s.proj=nn.Linear(C,C)
    def forward(s, x, cache, po):
        B,T,_=x.shape
        q=s.q(x).view(B,T,NH,HD).transpose(1,2); k=s.k(x).view(B,T,NH,HD).transpose(1,2); v=s.v(x).view(B,T,NH,HD).transpose(1,2)
        if cache is not None:
            if cache['k'] is not None: k=torch.cat([cache['k'],k],2); v=torch.cat([cache['v'],v],2)
            cache['k'],cache['v']=k,v
        Tk=k.size(2); att=(q@k.transpose(-2,-1))/math.sqrt(HD)
        qp=torch.arange(po,po+T).view(T,1); kp=torch.arange(Tk).view(1,Tk)
        att=att.masked_fill(kp>qp, float('-inf'))
        return s.proj((torch.softmax(att,-1)@v).transpose(1,2).reshape(B,T,C))
class Blk(nn.Module):
    def __init__(s): super().__init__(); s.ln1=nn.LayerNorm(C); s.ln2=nn.LayerNorm(C); s.at=Attn(); s.ff=nn.Sequential(nn.Linear(C,4*C),nn.GELU(),nn.Linear(4*C,C))
    def forward(s,x,c,po): x=x+s.at(s.ln1(x),c,po); return x+s.ff(s.ln2(x))
class GPT(nn.Module):
    def __init__(s): super().__init__(); s.tok=nn.Embedding(VOCAB,C); s.pos=nn.Embedding(BLOCK,C); s.blocks=nn.ModuleList([Blk() for _ in range(NL)]); s.lnf=nn.LayerNorm(C); s.head=nn.Linear(C,VOCAB)
    def forward(s, idx, caches=None, po=0):
        T=idx.size(1); x=s.tok(idx)+s.pos(torch.arange(po,po+T))
        for b,c in zip(s.blocks, caches or [None]*NL): x=b(x,c,po)
        return s.head(s.lnf(x))
    @torch.no_grad()
    def generate_naive(s, idx, n):
        for _ in range(n): idx=torch.cat([idx, s(idx[:,-BLOCK:])[:,-1].argmax(-1,keepdim=True)],1)
        return idx
    @torch.no_grad()
    def generate_cached(s, idx, n):
        caches=[{'k':None,'v':None} for _ in range(NL)]
        out=torch.cat([idx, s(idx,caches,0)[:,-1].argmax(-1,keepdim=True)],1)
        for _ in range(n-1): out=torch.cat([out, s(out[:,-1:],caches,out.size(1)-1)[:,-1].argmax(-1,keepdim=True)],1)
        return out

In [ ]:
torch.manual_seed(0); model = GPT().eval()
prompt = torch.randint(0, VOCAB, (1, 8))
a = model.generate_naive(prompt.clone(), 150)
b = model.generate_cached(prompt.clone(), 150)
print('identical output:', torch.equal(a, b))

for n in [128, 384]:
    t0=time.perf_counter(); model.generate_naive(prompt.clone(), n); tn=(time.perf_counter()-t0)*1000
    t0=time.perf_counter(); model.generate_cached(prompt.clone(), n); tc=(time.perf_counter()-t0)*1000
    print(f'{n} tokens: naive {tn:5.0f} ms | cached {tc:5.0f} ms | {tn/tc:.1f}x faster')

## Step 4 — The price: cache memory  ✍️ Your turn

The cache trades memory for speed: it stores a key AND a value per token, per layer — growing with
sequence length. Fill the formula and see a 7B model's cache dwarf its ~14 GB of weights.

<details><summary>Stuck? Show the answer</summary>

```python
return 2 * n_layers * n_heads * head_dim * seq_len * batch * bytes_per / 1e9
```
</details>

In [ ]:
def kv_cache_gb(n_layers, n_heads, head_dim, seq_len, batch=1, bytes_per=2):
    # ✍️ 2 (K and V) x layers x heads x head_dim x seq_len x batch x bytes, in GB
    return None      # replace

for ctx in [4096, 32768]:
    gb = kv_cache_gb(32, 32, 128, ctx)
    print(f'Llama-2-7B @ {ctx:>6} ctx: {gb if gb is None else f"{gb:.1f} GB"}')

In [ ]:
# ▶️ Check your work
try:
    gb = kv_cache_gb(32, 32, 128, 32768)
    if gb is None:
        print('⚠️  Fill in the ✍️ formula above, then re-run. (Expected until you do.)')
    elif abs(gb - 17.18) < 0.2:
        print(f'✅ {gb:.1f} GB of KV-cache at 32k context — bigger than the model\'s ~14 GB of weights!')
    else:
        print(f'⚠️  Got {gb:.1f} GB, expected ~17.2. Check the factors (2 x layers x heads x head_dim x seq x bytes).')
except Exception as e:
    print('❌', e)

## 🎉 You built a KV-cache.

You saw the O(T²) waste, proved the cache reproduces full attention exactly, watched a real GPT
generate identically but faster, and priced the memory. This is the trick behind every snappy chat
UI. Take it to your **GPT** in the [mini-project](../project/), then on to
[Chapter 13 — Quantization](../../13-inference-quantization/). ⚡